## Parsing and injesting the sea_marks datafile

In [1]:
import json
import os

os.chdir("../../..")

from tests.notebooks.graph_ingestion.neo4j_client import Neo4jClient

graph_client = Neo4jClient()
graph_client.delete_all_nodes()
graph_client.create_layers()

In [2]:
with open("raw_maps/seamarks_and_coastlines_solent.geojson") as map:
    raw_map = json.load(map)

conditional_processing = {
    "buoy_special_purpose": graph_client.injest_special_purpose_mark,
    "buoy_isolated_danger": graph_client.injest_isolated_danger_mark,
    "beacon_isolated_danger": graph_client.injest_isolated_danger_mark,
    "buoy_cardinal": graph_client.injest_cardinal_mark,
}

features = raw_map["features"]
cleaned_features = []
mark_count = 0
coastline_count = 0

print(f"Attempting to insert {len(features)} new marks.")

for feature in features:  
    try:
        seamark_type = feature["properties"]["seamark:type"]
    except KeyError:
        pass
    else:
        try:
            feature_extraction = conditional_processing[seamark_type](feature)
            mark_count += 1
        except KeyError as e:
            pass
        except Exception as e:
            print(f"Failed to inject a mark: {e}")
            raise
    

    # Coastline geoms
    try:
        if feature["properties"]["natural"] == "coastline":
            graph_client.injest_coastline(feature)
            coastline_count += 1
    except KeyError:
        pass

print(f"Successfully inserted {mark_count} new marks.")
print(f"Successfully inserted {coastline_count} new coastlines")

Attempting to insert 2074 new marks.
Successfully inserted 223 new marks.
Successfully inserted 379 new coastlines


---

### Create the edges

In [ ]:
edges = graph_client.create_safe_edges()

print(f"The legnth {len(edges)}")

[{'from': 'Hamble Point', 'to': None, 'distanceKm': 0.9033843481954169},
 {'from': 'Hamble Point', 'to': None, 'distanceKm': 0.9107592868319254},
 {'from': 'Hamble Point', 'to': 'Mark', 'distanceKm': 1.144152918368406},
 {'from': 'Hamble Point', 'to': 'Sposa', 'distanceKm': 1.5396801383676286},
 {'from': 'Hamble Point', 'to': 'Coronation', 'distanceKm': 1.655583514236405},
 {'from': 'Prince Consort',
  'to': 'Seafarers Ale',
  'distanceKm': 1.1890604361373394},
 {'from': 'Prince Consort', 'to': 'RORC', 'distanceKm': 1.3178281830207537},
 {'from': 'Prince Consort', 'to': 'Gurnard', 'distanceKm': 1.522651574032507},
 {'from': 'Prince Consort',
  'to': 'Deloitte Saling Club',
  'distanceKm': 1.6029943371928672},
 {'from': 'Prince Consort',
  'to': 'West Knoll',
  'distanceKm': 1.906863094970623},
 {'from': 'West Bramble', 'to': 'TeamO', 'distanceKm': 1.3311675101228206},
 {'from': 'West Bramble', 'to': 'Royal Cork', 'distanceKm': 1.461178735162272},
 {'from': 'West Bramble', 'to': 'Gurnar

---
#### Delete all nodes

In [5]:
graph_client.delete_all_nodes()

___
### Testing

In [4]:
from shapely.geometry import LineString, Polygon

def line_intersects_polygon(p1, p2, polygon_coords):
    """
    p1, p2: (lon, lat)
    polygon_coords: [(lon, lat), (lon, lat), ...]
    """
    line = LineString([p1, p2])
    polygon = Polygon(polygon_coords)
    return line.intersects(polygon)


# -------------------------
# Example usage
# -------------------------

# North test mark
point_a = (-0.9927808, 50.7189197)
# South test mark
point_b = (-0.9929109, 50.7174626)

# North test danger zone
danger_polygon = [(-0.9909808, 50.7170297),
(-0.9910153864952742, 50.717380862579624),
(-0.9911178168414797, 50.71771853017825),
(-0.9912841546978555, 50.71802972641943),
(-0.9915080077938643, 50.718302492206135),
(-0.9917807735805647, 50.718526345302145),
(-0.9920919698217429, 50.718692683158515),
(-0.9924296374203709, 50.718795113504726),
(-0.9927808, 50.7188297),
(-0.9931319625796291, 50.718795113504726),
(-0.9934696301782572, 50.718692683158515),
(-0.9937808264194353, 50.718526345302145),
(-0.9940535922061358, 50.718302492206135),
(-0.9942774453021446, 50.71802972641943),
(-0.9944437831585203, 50.71771853017825),
(-0.9945462135047258, 50.717380862579624),
(-0.9945808, 50.7170297),
(-0.9945462135047258, 50.71667853742037),
(-0.9944437831585203, 50.71634086982174),
(-0.9942774453021446, 50.716029673580564),
(-0.9940535922061358, 50.71575690779386),
(-0.9937808264194353, 50.71553305469785),
(-0.9934696301782572, 50.71536671684148),
(-0.9931319625796291, 50.71526428649527),
(-0.9927808, 50.715229699999995),
(-0.9924296374203709, 50.71526428649527),
(-0.9920919698217429, 50.71536671684148),
(-0.9917807735805647, 50.71553305469785),
(-0.9915080077938643, 50.71575690779386),
(-0.9912841546978555, 50.716029673580564),
(-0.9911178168414797, 50.71634086982174),
(-0.9910153864952742, 50.71667853742037),
(-0.9909808, 50.7170297)]

print(line_intersects_polygon(point_a, point_b, danger_polygon))


True
